# Swan — Data check for the questionnaire

## MVP scope decided on 23/10/2026

- **Concerne: **acne concern**
- **Output for MVP:** Moisturizers

## 📦 Data sources

| Source | File | Downloaded | Used here? |
|---|---|---|---|
| Open Beauty Facts | `raw_data/fr.openbeautyfacts.org.products.csv` | 25/10/2026 | ✅ Yes |
| EU Cosmetics Regulation (EC) 1223/2009 — consolidated 18.05.2026, Annex VI (UV filters) | `raw_data/eu_uv_filters_annex_vi.csv` (extracted from the PDF, ref + page per row) | 25/10/2026 | ✅ Yes |
| EU Cosmetics Regulation (EC) 1223/2009 — consolidated 18.05.2026, Annex III (fragrance allergens) | `raw_data/eu_fragrance_allergens_annex_iii.csv` (extracted from the PDF, ref + page per row) | 25/10/2026 | ✅ Yes |
| Open Beauty Facts — Parquet on Hugging Face (`openfoodfacts/product-database`, `beauty.parquet`, 64.4 MB) | — | — | ❌ Not used — alternative source to discuss with Lina |
| EU glossary of common ingredient names — [Decision (EU) 2025/1175](https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=OJ:L_202501175) | — | — | ❌ No (useful for J1, INCI normalisation) |
| CosIng — EU cosmetic ingredient database | — | — | ❌ Not yet (J1, tasks T09 / T11) |


## 🎯  Questionnaire for MVP 
| # | Question | Profile field | Options shown → stored value | Required | Action | Based on |
|---|---|---|---|---|---|---|
| 1 | Are you 18 or older? | `is_adult` | yes → `true` / no → `false` | ✅ | refer if `false` | scope rule |
| 2 | What would you like help with? | `concern` | a simple routine for acne-prone skin → `acne_prone` / something else → `other` | ✅ | out-of-scope message if `other` | MVP scope  |
| 3 | How would you describe your blemishes? | `blemish_level` | rank from 1 to 5 | ✅ |  | referral rule | Validated on 26/10
| 4 | Does any of these apply to you? | `referral_flags` | prescribed treatment → `prescribed_treatment` / sudden or unusual lesion → `unusual_lesion` / pregnant or breastfeeding → `pregnant_or_breastfeeding` / none of these → empty list | ✅ | refer if not empty | referral rule |
| 5 | What is your skin type? | `skin_type` | `oily` / `combination` / `normal` / `dry` / not sure → `not_sure` | ✅ | rank | data to review |
| 6 | Does your skin react easily (stinging, redness)? | `reactivity` | `never` / `sometimes` / `often` | ✅ | rank (penalise fragrance) | data to review | GO with Yes or No for the MVP
| 7 | Ingredients you prefer to avoid? | `avoid` | fragrance → `fragrance` / essential oils → `essential_oils` / none → empty list | ❌ | filter 
| 8 | Which texture do you prefer? | `texture` | light → `light` / rich → `rich` / no preference → `no_preference` | ❌ (default `no_preference`) | NOT FOR THE MVP 


## To validate on 26/10

| # | Decision | Proposal | Decision |
|---|---|---|---|
| B02 | MVP | Adults with acne-prone skin, mild blemishes · cleanser + moisturizer + facial sunscreen · Number of products we need in each category (cleanser, moisturizer, sunscreen).| 1 issue acne - output moisturizer
| B04 | Market scope | EU (EU + EEA as an option: the regulation has "EEA relevance") · France is the core of the data | Start with France as a start
| — | Role edge cases | SPF moisturizer stays a moisturizer (sunscreen slot = dedicated sunscreens) · toners out · micellar water = cleanser · face lotions after examples · anti-ageing out | To split into to different categories moisterizer and sunscreen based on the tags.
| T12 | Questionnaire v1 + profile schema | 8 questions, safety first 
| T23 | Test-profile grid | Built from the profile fields; first 6 profiles in section 11
| T27 | Mockups | 4 screens: welcome, questionnaire, routine, referral 
| T06 | Data source | Same snapshot for the whole team (this CSV or `beauty.parquet`)

**File facts** :
- Number of lines : 64 238 
- About 200 columns, most of them food nutrition columns (`*_100g`) inherited from Open Food Facts
- First rows show noise: a food product
- We have cream with SPF should this be flagged as moisterizer or sunscream or should we have a mention for it for a substitute? 
- We have all countries but most of the data is in EU - What should we do ? Focus on European Countries ? 


## Settings to run my notebook

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [38]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

FILE_PATH = "../raw_data/fr.openbeautyfacts.org.products.csv"
UV_FILTERS_PATH = "../raw_data/eu_uv_filters_annex_vi.csv"
ALLERGENS_PATH = "../raw_data/eu_fragrance_allergens_annex_iii.csv"
LINES_IN_FILE = 64238

SEED = 42
NB_ROUTINE_PRODUCT = 3

EU_COUNTRY_TAGS = [
    "en:france", "en:belgium", "en:germany", "en:spain", "en:italy", "en:netherlands",
    "en:luxembourg", "en:portugal", "en:austria", "en:ireland", "en:greece", "en:poland",
    "en:czech-republic", "en:czechia", "en:slovakia", "en:hungary", "en:romania", "en:bulgaria",
    "en:croatia", "en:slovenia", "en:denmark", "en:sweden", "en:finland", "en:estonia",
    "en:latvia", "en:lithuania", "en:cyprus", "en:malta",
]
EEA_EXTRA_TAGS = ["en:norway", "en:iceland", "en:liechtenstein"]
COUNTRY_TAGS = EU_COUNTRY_TAGS

print(FILE_PATH)

../raw_data/fr.openbeautyfacts.org.products.csv


## Load Data Set and check the columns

In [3]:
df_complete_file = pd.read_csv(FILE_PATH, sep="\t", dtype={"code": str})
df_complete_file.shape

/var/folders/bl/bvf3l2w16sb0jmy2v926fp0h0000gn/T/ipykernel_28223/1418356569.py:1: DtypeWarning: Columns (11,34,36,54,60) have mixed types. Specify dtype option on import or set low_memory=False.
  df_complete_file = pd.read_csv(FILE_PATH, sep="\t", dtype={"code": str})


(64237, 200)

In [4]:
df_complete_file.head()

,code,url,creator,created_t,created_datetime,last_modified_t,last_modified_datetime,last_modified_by,last_updated_t,last_updated_datetime,...,choline_100g,phylloquinone_100g,beta-glucan_100g,inositol_100g,carnitine_100g,sulphate_100g,nitrate_100g,acidity_100g,carbohydrates-total_100g,water_100g
0,00000001,http://world-en.openbeautyfacts.org/product/00...,tonyracket,1751609682,2025-07-04T06:14:42Z,1751609686,2025-07-04T06:14:46Z,tonyracket,1.751610e+09,2025-07-04T06:14:46Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00000014,http://world-en.openbeautyfacts.org/product/00...,insectproductadd,1710345659,2024-03-13T16:00:59Z,1748277440,2025-05-26T16:37:20Z,fitcounter,1.748277e+09,2025-05-26T16:37:20Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00000092,http://world-en.openbeautyfacts.org/product/00...,product-scan-com,1638580447,2021-12-04T01:14:07Z,1744901851,2025-04-17T14:57:31Z,smoothie-app,1.744902e+09,2025-04-17T14:57:31Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00000163,http://world-en.openbeautyfacts.org/product/00...,smoothie-app,1682257711,2023-04-23T13:48:31Z,1727959822,2024-10-03T12:50:22Z,fix-code-bot,1.743167e+09,2025-03-28T13:01:25Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00000280,http://world-en.openbeautyfacts.org/product/00...,date-limite-app,1584379762,2020-03-16T17:29:22Z,1744884567,2025-04-17T10:09:27Z,smoothie-app,1.744885e+09,2025-04-17T10:09:27Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df_complete_file["completeness"].describe()

count    56361.000000
mean         0.310339
std          0.220251
min          0.050000
25%          0.075000
50%          0.275000
75%          0.462500
max          1.100000
Name: completeness, dtype: float64

In [6]:
# products created per year
df_complete_file["created_datetime"].fillna("").str[:4].value_counts().sort_index()

created_datetime
2013        6
2014      148
2015      244
2016     2556
2017     1736
2018     5153
2019     3953
2020     4359
2021     5865
2022     4967
2023     5943
2024     8437
2025    16391
2026     4479
Name: count, dtype: int64

In [7]:
df_complete_file.dtypes.value_counts()

float64    132
object      66
int64        2
Name: count, dtype: int64

## Do we have all the fields needed complete?

In [8]:
states = df_complete_file["states_tags"].fillna("")
for part in ["ingredients", "categories", "product-name", "brands"]:
    done = states.str.contains("en:" + part + "-completed").sum()
    print(f"{part:14} completed: {done:6} ({round(done / len(df_complete_file) * 100, 1)} %)")

ingredients    completed:  17303 (26.9 %)
categories     completed:  28496 (44.4 %)
product-name   completed:  40029 (62.3 %)
brands         completed:  34549 (53.8 %)


In [9]:
#Check if we have all EU countries in the dataset
countries = df_complete_file["countries_tags"].fillna("")
single_country = countries[(countries != "") & ~countries.str.contains(",")]
single_country.value_counts().head(40)

countries_tags
en:france                  19893
en:germany                  4620
en:united-states            3947
en:morocco                  3782
en:spain                    2961
en:united-kingdom           2079
en:italy                    1436
en:india                    1399
en:netherlands              1391
en:portugal                  958
en:canada                    953
en:norway                    926
en:russia                    805
en:switzerland               683
en:belgium                   662
en:algeria                   607
en:saudi-arabia              600
en:romania                   543
en:brazil                    441
en:poland                    376
en:ukraine                   348
en:sweden                    347
en:turkey                    346
en:argentina                 346
en:austria                   315
en:philippines               313
en:mexico                    310
en:australia                 306
en:united-arab-emirates      306
en:colombia                 

In [10]:
# which countries appear most?
for tag in EU_COUNTRY_TAGS + ["en:united-kingdom", "en:switzerland", "en:united-states"]:
    n = countries.str.contains(tag).sum()
    if n > 0:
        print(f"{tag:22} {n}")

en:france              21780
en:belgium             762
en:germany             5017
en:spain               3224
en:italy               1519
en:netherlands         1452
en:luxembourg          79
en:portugal            990
en:austria             403
en:ireland             187
en:greece              258
en:poland              390
en:czech-republic      118
en:slovakia            62
en:hungary             93
en:romania             588
en:bulgaria            206
en:croatia             107
en:slovenia            34
en:denmark             78
en:sweden              375
en:finland             100
en:estonia             47
en:latvia              52
en:lithuania           127
en:cyprus              60
en:malta               20
en:united-kingdom      2206
en:switzerland         804
en:united-states       4197


## Group columns into families
Group the column into families based on the ones we will use for the questionnaire and for our model.

In [11]:
def column_family(name):
    """Return the family of a column, based on its name."""
    if name.endswith("_100g"):
        return "food nutrition"
    elif "image" in name:
        return "images"
    elif "ingredient" in name:
        return "ingredients"
    elif "categor" in name:
        return "categories"
    elif "label" in name:
        return "labels"
    elif "countr" in name:
        return "countries"
    elif name in ["code", "url", "product_name", "abbreviated_product_name", "generic_name", "quantity"] or "brand" in name:
        return "identity"
    elif "state" in name or "complet" in name or "quality" in name or "created" in name or "modified" in name or "creator" in name:
        return "quality"
    else:
        return "other"

In [12]:
families = pd.Series(df_complete_file.columns).apply(column_family)
families.index = df_complete_file.columns
families.value_counts()

food nutrition    123
other              34
quality            11
identity           10
images              8
categories          5
labels              3
countries           3
ingredients         3
Name: count, dtype: int64

## What should we take main category or category tags ?

In [13]:
# check the categories by iteration
df_complete_file["main_category"].value_counts().head(20)

main_category
en:open-beauty-facts          4271
en:cosmetic-products          3937
en:shampoos                   1051
en:shower-gels                 687
en:deodorants                  505
en:toothpastes                 492
en:sunscreen                   410
en:soaps                       384
en:non-open-products-facts     339
nl:verzorging                  318
en:hand-creams                 261
en:toothpaste                  197
en:lip-balms                   160
en:roll-on-deodorants          153
en:body-milks                  142
en:body-creams                 134
en:anti-dandruff-shampoos      128
en:hair-care                   126
nl:skincare                    125
fr:cremes-douche               123
Name: count, dtype: int64

In [14]:
#check the category tags if this is better to use
total = len(df_complete_file)
generic = df_complete_file["main_category"].isin(["en:open-beauty-facts", "en:cosmetic-products"]).sum()
empty = df_complete_file["main_category"].isnull().sum()
top_20 = df_complete_file["main_category"].value_counts().head(20).sum()

print("All products          :", total)
print("Generic main category :", generic, f"({round(generic / total * 100, 1)} %)")
print("Empty main category   :", empty, f"({round(empty / total * 100, 1)} %)")
print("Top 20 covers         :", top_20, f"({round(top_20 / total * 100, 1)} %)")

All products          : 64237
Generic main category : 8208 (12.8 %)
Empty main category   : 38589 (60.1 %)
Top 20 covers         : 13943 (21.7 %)


In [15]:
# same keywords, two columns: which one finds more of our products?
for word in ["cleans", "moistur", "sunscreen"]:
    in_main = df_complete_file["main_category"].fillna("").str.contains(word, case=False).sum()
    in_tags = df_complete_file["categories_tags"].fillna("").str.contains(word, case=False).sum()
    print(f"{word:10} main_category: {in_main:5}   categories_tags: {in_tags:5}")

cleans     main_category:   127   categories_tags:   277
moistur    main_category:   100   categories_tags:   103
sunscreen  main_category:   430   categories_tags:   467


## Check the products we have in the datase 

In [ ]:
products = df_complete_file[df_complete_file["countries_tags"].fillna("").str.contains("|".join(COUNTRY_TAGS))].copy()
print("All products            :", len(df_complete_file))
print("In scope                :", len(products))

products = products[products["ingredients_text"].str.strip().str.len() >= 20].copy()
print("+ usable ingredient list:", len(products))

All products            : 64237
In scope                : 37419
+ usable ingredient list: 11419


In [16]:
total = len(df_complete_file)
generic = df_complete_file["main_category"].isin(["en:open-beauty-facts", "en:cosmetic-products"]).sum()
empty = df_complete_file["main_category"].isnull().sum()
top_20 = df_complete_file["main_category"].value_counts().head(20).sum()

print("All products          :", total)
print("Generic main category :", generic, f"({round(generic / total * 100, 1)} %)")
print("Empty main category   :", empty, f"({round(empty / total * 100, 1)} %)")
print("Top 20 covers         :", top_20, f"({round(top_20 / total * 100, 1)} %)")

All products          : 64237
Generic main category : 8208 (12.8 %)
Empty main category   : 38589 (60.1 %)
Top 20 covers         : 13943 (21.7 %)


In [17]:
# same keywords, two columns: which one finds more of our products?
for word in ["cleans", "moistur", "sunscreen"]:
    in_main = df_complete_file["main_category"].fillna("").str.contains(word, case=False).sum()
    in_tags = df_complete_file["categories_tags"].fillna("").str.contains(word, case=False).sum()
    print(f"{word:10} main_category: {in_main:5}   categories_tags: {in_tags:5}")

cleans     main_category:   127   categories_tags:   277
moistur    main_category:   100   categories_tags:   103
sunscreen  main_category:   430   categories_tags:   467


**Question:** which products are cleansers, moisturizers, sunscreens — and validate the rule I used? 

In [26]:
def find_role(categories):
    """Return the product role from its categories."""
    text = str(categories).lower()
    if "anti-wrinkle" in text or "anti-aging" in text or "anti-age" in text:
        return "other"                                   # ageing concern: outside the MVP
    elif "facial-cream" in text or "face-cream" in text or "cremes-hydratantes" in text or "creme-visage" in text:
        return "moisturizer"                             # clear face moisturizers, before the exclusions
    elif "hair" in text or "shampoo" in text or "body" in text or "hand" in text or "shower" in text or "tooth" in text or "after-sun" in text:
        return "other"
    elif "sunscreen" in text or "sun-protection" in text or "solaire" in text:
        return "sunscreen"
    elif "cleans" in text or "nettoyant" in text:
        return "cleanser"
    elif "moistur" in text or "day-cream" in text:
        return "moisturizer"
    else:
        return "other"

products["role"] = products["categories_tags"].apply(find_role)
products["role"].value_counts()

role
other          10813
sunscreen        244
moisturizer      202
cleanser         160
Name: count, dtype: int64

**Check** check the name based on the product name

In [27]:
def find_role_from_name(name):
    """Guess the role from words in the product name (English and French)."""
    text = str(name).lower()
    if "spf" in text or "sunscreen" in text or "solaire" in text or "sun cream" in text:
        return "sunscreen"
    elif "cleanser" in text or "nettoyant" in text or "micellar" in text or "micellaire" in text or "face wash" in text:
        return "cleanser"
    elif "moisturi" in text or "hydratant" in text or "day cream" in text or "crème visage" in text or "creme visage" in text:
        return "moisturizer"
    else:
        return "other"

products["name_role"] = products["product_name"].apply(find_role_from_name)
pd.crosstab(products["role"], products["name_role"])

name_role,cleanser,moisturizer,other,sunscreen
role,,,,
cleanser,46,3,111,0
moisturizer,4,43,149,6
other,160,243,10275,135
sunscreen,0,4,133,107


In [28]:
others = products[products["role"] == "other"]
face_words = "face|facial|visage|skin-care|creme|cream|soin"
face_in_other = others[others["categories_tags"].fillna("").str.contains(face_words, case=False)]

print("'Other' products with face-care words:", len(face_in_other))
face_in_other["main_category"].value_counts().head(25)

'Other' products with face-care words: 991


main_category
en:hand-creams                        148
fr:cremes-douche                       97
en:lip-cosmetics                       43
en:face-masks                          37
en:body-creams                         34
en:open-beauty-facts                   25
en:anti-wrinkles-creams                20
en:face                                19
nl:beauty                              18
nl:skincare                            17
en:lip-balms                           17
ar:كريمات-مرطبة                        17
en:face-lotions                        16
fr:baumes-et-beurres-pour-le-corps     16
en:face-scrubs                         14
nl:verzorging                          12
en:foundations                         12
en:day-creams                          10
fr:cremes-pour-les-pieds               10
en:dairies                              9
de:handcreme                            9
en:anti-aging-face-care-products        9
en:shower-gels                          9
fr:contours-des-yeux

In [29]:
face_words = "face|facial|visage|skin-care|creme|cream|soin"
others = products[products["role"] == "other"]
face_others = others[others["categories_tags"].fillna("").str.contains(face_words, case=False)]
print("'Other' products with face-care words:", len(face_others))

miss_check = face_others.sample(min(20, len(face_others)), random_state=SEED)
for name, url in zip(miss_check["product_name"], miss_check["url"]):
    print(name, "|", url)

'Other' products with face-care words: 991
Crème mains réparatrice peaux très sèches | http://world-en.openbeautyfacts.org/product/3350033437791/creme-mains-reparatrice-peaux-tres-seches-monoprix
Inell crème de soin visage et corps | http://world-en.openbeautyfacts.org/product/3564700750223/inell-creme-de-soin-visage-et-corps-inell-leclerc
Baume à lèvres 040 rose blush | http://world-en.openbeautyfacts.org/product/3607343722278/baume-a-levres-040-rose-blush-rimmel
Soin gommant corne & callosités | http://world-en.openbeautyfacts.org/product/3560070231584/soin-gommant-corne-callosites-carrefour
Lait de toilette très doux Vitamine E Huile d'amande douce | http://world-en.openbeautyfacts.org/product/3058320006001/lait-de-toilette-tres-doux-vitamine-e-huile-d-amande-douce-mixa-bebe
1er Sérum physiologique naturel | http://world-en.openbeautyfacts.org/product/3661434000546/1er-serum-physiologique-naturel-uriage
Hydra AloeVera | http://world-en.openbeautyfacts.org/product/3517360013771/hydra

In [31]:
for role in ["cleanser", "moisturizer", "sunscreen"]:
    pool_role = products[products["final_role"] == role]
    check = pool_role.sample(min(10, len(pool_role)), random_state=SEED)
    print("====", role)
    for name, url in zip(check["product_name"], check["url"]):
        print(name, "|", url)

==== cleanser
Eau nettoyante | http://world-en.openbeautyfacts.org/product/3350033362123/eau-nettoyante-bout-chou
Eau Pure Nettoyante H20 Pour Bébés Biolane, | http://world-en.openbeautyfacts.org/product/3286018000288/eau-pure-nettoyante-h20-pour-bebes-biolane
Neostrata Skin Active Mousse Exfoliante Nettoyante 125 ML | http://world-en.openbeautyfacts.org/product/3401320353857/neostrata-skin-active-mousse-exfoliante-nettoyante-125-ml
Nettoyant Multi-usages Spray écologique Pamplemousse Rose ? ? Method? | http://world-en.openbeautyfacts.org/product/0817939012475/nettoyant-multi-usages-spray-ecologique-pamplemousse-rose-method-jean-mi-de-paris
Gel nettoyant pour les mains sans savon | http://world-en.openbeautyfacts.org/product/3564700514542/gel-nettoyant-pour-les-mains-sans-savon-marque-repere
Gel douche micellaire sans savon Thé blanc & camomille | http://world-en.openbeautyfacts.org/product/3346029200081/gel-douche-micellaire-sans-savon-the-blanc-camomille-vendome
Solution Micellaire T

**Check 2 — missed products:** 20 random "other" products with face-care words to check how many are moisterizers or cleansers or sunscreen

In [32]:
face_words = "face|facial|visage|skin-care|creme|cream|soin"
others = products[products["final_role"] == "other"]
face_others = others[others["categories_tags"].fillna("").str.contains(face_words, case=False)]
print("'Other' products with face-care words:", len(face_others))
face_others["main_category"].value_counts().head(15)

'Other' products with face-care words: 888


main_category
en:hand-creams                        142
fr:cremes-douche                       85
en:lip-cosmetics                       37
en:face-masks                          35
en:body-creams                         27
en:anti-wrinkles-creams                19
nl:beauty                              17
en:open-beauty-facts                   17
en:lip-balms                           17
fr:baumes-et-beurres-pour-le-corps     15
nl:skincare                            14
en:face-scrubs                         14
en:face-lotions                        13
ar:كريمات-مرطبة                        12
en:face                                12
Name: count, dtype: int64

## 9. Edge cases: SPF moisturizers, toners, face lotions, micellar waters

**Question:** is a moisturizer with SPF / UV filters a sunscreen? Are toners, face lotions, micellar waters cleansers, moisturizers or out? ("SPF 50" is the **claim**; UV filters are the **ingredients** that provide it.)

## 10. Routine products and texture

**Question:** can we detect texture for Q5 (skin type) and Q8 (texture preference)? There is no texture column: we look for texture words.

In [36]:
routine = products[products["final_role"] != "other"].copy()

text = routine["product_name"].fillna("") + " " + routine["categories_tags"].fillna("")
routine["is_light"] = text.str.contains("gel|fluid|fluide|foam|mousse|lotion", case=False)
routine["is_rich"] = text.str.contains("cream|creme|crème|balm|baume", case=False)
routine.groupby("final_role")[["is_light", "is_rich"]].sum()

,is_light,is_rich
final_role,,
cleanser,75,9
moisturizer,64,297
sunscreen,38,75


In [37]:
((routine["is_light"] | routine["is_rich"]).groupby(routine["final_role"]).mean() * 100).round(1)

final_role
cleanser       25.3
moisturizer    78.0
sunscreen      28.8
dtype: float64